# NGEO Satellite Downloader

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/ngeo_satellite_downloader/blob/main/ngeo_satellite_downloader.ipynb)

Ferramenta do Nucleo de Geotecnologias (NGEO) do Instituto de Desenvolvimento Florestal e da Biodiversidade do Estado do Para (IDEFLOR-Bio) para aquisicao e download automatizado de imagens de satelite sobre areas delimitadas por shapefile.

### Sensores Orbitais Disponiveis

| Sensor | Resolucao Espacial | Caracteristicas e Aplicacoes |
|---|---:|---|
| Sentinel-2 MSI | 10 m | Composicao semestral ou mensal com mascara de nuvens via classificacao SCL. |
| Landsat 5, 7, 8 e 9 | 30 m | Series historicas calibradas em reflectancia de superficie com mascara QA_PIXEL. |
| CBERS-4A (WPM e MUX) | 2 m | Imagens nacionais de alta resolucao espacial integradas a API do INPE com composicao RGBN. |
| SPOT 2008 | 5 m | Mosaico de referencia historica do Codigo Florestal Brasileiro. |

### Roteiro de Operacao
1. Instalacao de dependencias e preparacao dos modulos tecnicos.
2. Autenticacao no Google Earth Engine e montagem da unidade de armazenamento Google Drive.
3. Carregamento do arquivo ZIP contendo o shapefile da area de interesse.
4. Definicao do sensor, intervalo cronologico, metodo de composicao e buffer de seguranca.
5. Execucao do fluxo de amostragem e submissao das tarefas de exportacao em GeoTIFF.

In [ ]:
#@title 1. Instalacao de Dependencias e Preparacao do Ambiente { display-mode: "form" }
#@markdown *Execute esta celula para instalar as bibliotecas geoespaciais e carregar o pacote tecnico do NGEO.*

!pip install -q geemap cbers4asat rasterio geopandas shapely pycrs geomet geojson scikit-image requests pillow

import os
import sys
from pathlib import Path

REPO_NAME = "ngeo_satellite_downloader"

# Garantir clonagem ou atualizacao automatica com a branch principal
if Path(REPO_NAME).exists():
    !git -C {REPO_NAME} pull origin main --quiet
    os.chdir(REPO_NAME)
elif not Path("src").exists():
    print(f"Clonando repositorio oficial {REPO_NAME}...")
    !git clone https://github.com/NGEO-IDEFLOR-Bio/ngeo_satellite_downloader.git
    os.chdir(REPO_NAME)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import importlib
import src
import src.vector_ops
import src.sensors
import src.cbers
import src.exporter
import src.ui_helpers

importlib.reload(src)
importlib.reload(src.vector_ops)
importlib.reload(src.sensors)
importlib.reload(src.cbers)
importlib.reload(src.exporter)
importlib.reload(src.ui_helpers)

print(f"[OK] Ambiente operacional configurado. Versao: {src.__version__}")


In [ ]:
#@title 2. Autenticacao no Google Earth Engine e Google Drive { display-mode: "form" }
GCP_PROJECT = "sentinel-landsat-download" #@param {type:"string"}
MONTAR_GOOGLE_DRIVE = True #@param {type:"boolean"}
#@markdown *Nota: Informe o identificador do projeto Google Cloud autorizado no Earth Engine.*

import os
import ee
from google.colab import drive

try:
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Google Earth Engine inicializado no projeto: {GCP_PROJECT}")
except Exception as e:
    print("Solicitando autenticacao interativa no Google Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Earth Engine inicializado no projeto: {GCP_PROJECT}")

if MONTAR_GOOGLE_DRIVE:
    try:
        drive.mount('/content/drive')
        print("[OK] Google Drive montado em /content/drive")
    except Exception as e:
        print(f"[AVISO] Nao foi possivel montar o Google Drive: {e}")


In [ ]:
#@title 3. Carregamento da Area de Interesse (Shapefile) { display-mode: "form" }
#@markdown *Envie um arquivo compactado em formato ZIP contendo os arquivos .shp, .shx, .dbf e .prj.*
#@markdown *O processamento ajusta as coordenadas para o sistema WGS-84 (EPSG:4326) e higieniza geometrias 2D.*

import os
import io
import shutil
import importlib
from google.colab import files
import src.vector_ops
importlib.reload(src.vector_ops)
from src.vector_ops import extract_shapefile_from_zip, load_shapefile_geometry, calculate_aoi_stats
from src.ui_helpers import render_aoi_card

print("Selecione o arquivo ZIP contendo o shapefile:")
uploaded = files.upload()

SHAPEFILE_GEOMETRY = None
GDF_AOI = None
AOI_STATS = None

if uploaded:
    zip_name = list(uploaded.keys())[0]
    zip_bytes = uploaded[zip_name]
    shp_path = extract_shapefile_from_zip(zip_bytes, extract_dir="shapefile")
    print(f"[OK] Shapefile extraido: {shp_path}")
    
    SHAPEFILE_GEOMETRY, GDF_AOI = load_shapefile_geometry(shp_path)
    AOI_STATS = calculate_aoi_stats(SHAPEFILE_GEOMETRY, GDF_AOI)
    render_aoi_card(AOI_STATS)
else:
    shp_fallback = [os.path.join("shapefile", f) for f in os.listdir("shapefile") if f.endswith(".shp")] if os.path.exists("shapefile") else []
    if shp_fallback:
        print(f"[INFO] Utilizando shapefile preexistente: {shp_fallback[0]}")
        SHAPEFILE_GEOMETRY, GDF_AOI = load_shapefile_geometry(shp_fallback[0])
        AOI_STATS = calculate_aoi_stats(SHAPEFILE_GEOMETRY, GDF_AOI)
        render_aoi_card(AOI_STATS)
    else:
        print("[AVISO] Nenhum arquivo foi enviado. Execute novamente a celula para selecionar o shapefile.")


In [ ]:
#@title 3.1 (Opcional) Visualizacao Cartografica da Area de Interesse { display-mode: "form" }
#@markdown *Renderiza mapa interativo para confirmacao visual dos limites territoriais informados.*

import geemap
from IPython.display import display

if "SHAPEFILE_GEOMETRY" in globals() and SHAPEFILE_GEOMETRY is not None:
    centro = AOI_STATS.get("center", [-3.5, -52.0]) if AOI_STATS else [-3.5, -52.0]
    Map = geemap.Map(center=centro, zoom=10)
    Map.add_basemap("SATELLITE")
    Map.addLayer(SHAPEFILE_GEOMETRY, {"color": "red", "fillColor": "00000000", "width": 2}, "Area de Interesse")
    display(Map)
else:
    print("[AVISO] Execute previamente a etapa 3 para carregar a geometria da area.")


In [ ]:
#@title 4. Parametros Operacionais e Selecao do Sensor { display-mode: "form" }
SATELITE = "Sentinel-2 (10m)" #@param ["Sentinel-2 (10m)", "Landsat 5/7/8/9 (30m)", "CBERS-4A (MUX/WPM 2m)", "SPOT 2008 (Codigo Florestal 5m)"]
ANOS = "2024" #@param {type:"string"}
#@markdown *Formatos aceitos: ano unico (ex: 2024), intervalo (ex: 2020-2024) ou lista (ex: 2020, 2022, 2024).*

MODO = "Composicao por Semestre" #@param ["Composicao por Semestre", "Analise Anual (Meses Selecionados)"]
SEMESTRE = "Ambos" #@param ["1 Semestre (Jan a Jun)", "2 Semestre (Jul a Dez)", "Ambos"]
MESES_ANALISE = "6, 7, 8" #@param {type:"string"}
#@markdown *Para analise mensal, informe os numeros dos meses separados por virgula (ex: 6, 7, 8 para junho, julho e agosto).*

METODO = "median" #@param [("Mediana estatistica (redutor de ruidos e nuvens)", "median"), ("Melhor cena (menor indice de nebulosidade na area)", "best")]
BUFFER = 2 #@param {type:"slider", min:1.0, max:5.0, step:0.5}
#@markdown *Fator multiplicador da extensao envolvente (1.0 = limites exatos; 2.0 = margem operacional recomendada).*

PASTA_DRIVE = "Satellite_Downloads" #@param {type:"string"}
#@markdown *Subpasta no Google Drive onde os arquivos GeoTIFF serao depositados.*

INPE_EMAIL = "" #@param {type:"string"}
INPE_SENHA = "" #@param {type:"raw"}
#@markdown *Credenciais do catalogo INPE requeridas exclusivamente para aquisicoes CBERS-4A.*

from src.vector_ops import parse_years

anos_lista = parse_years(ANOS)
meses_lista = [int(m.strip()) for m in MESES_ANALISE.split(',') if m.strip().isdigit()]

print("Parametros configurados:")
print(f"Sensor: {SATELITE}")
print(f"Anos: {anos_lista}")
print(f"Modo: {MODO}")
if MODO == "Composicao por Semestre":
    print(f"Semestre: {SEMESTRE}")
else:
    print(f"Meses: {meses_lista}")
print(f"Metodo: {METODO}")
print(f"Fator de buffer: {BUFFER}x")
print(f"Pasta Google Drive: {PASTA_DRIVE}")


In [ ]:
#@title 5. Execucao do Processamento e Exportacao { display-mode: "form" }
#@markdown *Inicia o processamento espectral, aplicacao de mascaras e submissao das tarefas ao Earth Engine.*

import sys
from src.exporter import run_pipeline

if "SHAPEFILE_GEOMETRY" not in globals() or SHAPEFILE_GEOMETRY is None:
    raise ValueError("Execute primeiramente a etapa 3 para carregar a geometria da area.")

resultado = run_pipeline(
    region=SHAPEFILE_GEOMETRY,
    satellite=SATELITE,
    years=anos_lista,
    mode=MODO,
    semester=SEMESTRE,
    months=meses_lista,
    method=METODO,
    buffer_factor=float(BUFFER),
    drive_folder=PASTA_DRIVE,
    inpe_email=INPE_EMAIL.strip() if INPE_EMAIL else None
)


In [ ]:
#@title 6. Relatorio de Exportacao e Acesso aos Arquivos { display-mode: "form" }

from src.ui_helpers import render_results_cards

if "resultado" in globals() and resultado:
    render_results_cards(resultado)
else:
    print("[AVISO] Nenhuma operacao foi executada ainda. Execute a etapa 5.")


### Controle de Qualidade e Normas Tecnicas
1. Inspecao no software QGIS para conferencia de datum e alinhamento geografico.
2. Inspecao de artefatos de nebulosidade residuais em series temporais amazonicas.
3. Verificacao da continuidade espacial nas areas que exigiram particionamento em tiles.

Desenvolvimento e coordenacao: Samuel da Costa dos Santos (samuelsantos.site).
Nucleo de Geotecnologias (NGEO), Instituto de Desenvolvimento Florestal e da Biodiversidade do Estado do Para (IDEFLOR-Bio).